# MViT

**0 前言**<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.1 为什么需要MViT？<br>
&nbsp;&nbsp;&nbsp;&nbsp;0.2 MViT想解决ViT的什么问题？<br>

**1 MViT的核心思想与整体结构**<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.1 什么叫多尺度Transformer？<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.2 MViT整体结构与数据流<br>
&nbsp;&nbsp;&nbsp;&nbsp;1.3 MViT和ViT的关键区别<br>

<font color="red">**2 MViT中的尺度变化机制**</font><br>
&nbsp;&nbsp;&nbsp;&nbsp;2.1 为什么视觉建模需要多尺度表示？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.2 Token数量为什么要逐步减少？<br>
&nbsp;&nbsp;&nbsp;&nbsp;2.3 通道维度为什么要逐步增加？<br>

**3 MViT中的注意力是如何配合尺度变化的**<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.1 Pooling Attention的基本思路<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.2 下采样后注意力计算发生了什么变化？<br>
&nbsp;&nbsp;&nbsp;&nbsp;3.3 这样设计为什么更适合视觉任务？<br>

**4 MViT的特点、优势与理解要点**<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.1 MViT相比ViT解决了什么问题？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.2 MViT和CNN在多尺度思想上有什么相似与不同？<br>
&nbsp;&nbsp;&nbsp;&nbsp;4.3 学习MViT时最容易混淆的点<br>


## 2 MViT中的尺度变化机制

上一节已经把 MViT 和 ViT 的关键区别压缩成一句话：**ViT 更像在固定尺度上加工 token，MViT 则让 token 在网络内部逐步改变尺度。**

但这时还有一个更根本的问题没有展开：

**为什么视觉建模这件事，本来就不应该只停留在单一尺度上？**

这一节就专门回答这个问题。先从“为什么需要多尺度表示”讲起，再进一步讨论：

- 为什么 token 数量不能一直保持很多
- 为什么通道维度又要随着层数逐步增加

也就是说，`2.1` 先解决“为什么必须有尺度变化”，后面的 `2.2` 和 `2.3` 再分别解释这种变化在数量和通道两个方向上是怎样展开的。

### 2.1 为什么视觉建模需要多尺度表示？

先回忆一个最直观的现象：我们看一幅图像，或者看一段视频，并不是始终用同样精细的粒度在理解它。

比如看一段“一个人从远处跑来并挥手”的视频时，前面可能更需要看清局部运动边缘、衣服纹理、手臂摆动这些细节；但当模型继续往后理解时，又需要把这些局部线索组合起来，逐渐形成“这是一个人”“他正在跑”“最后做了挥手动作”这样的更大范围语义。

这说明视觉理解天然就有一个层次：

- 低层更关心局部、细粒度、短距离的信息
- 高层更关心整体、抽象、长范围的信息

如果一个模型从头到尾都停留在单一尺度上，就会很别扭。

为什么这么说？因为单一尺度通常会落入两种都不理想的情况。

第一种情况是：**一直保持很细的尺度。**

这样做的好处是局部细节保留得多，但问题也很明显：token 会非常多。图像还勉强能承受，到了视频里，时间维一展开，时空位置数量会迅速膨胀。这不仅计算代价高，而且模型后层仍然被迫在海量细粒度 token 上做建模，不利于把信息收束成更稳定的高层语义。

第二种情况是：**一开始就把尺度做得很粗。**

这样 token 数量会小很多，计算也更省，但代价是模型太早丢掉了局部结构。很多细小动作、边缘变化、局部外观差异，本来应该在前层先被看清，结果还没来得及充分建模，就已经被粗糙汇总掉了。

所以，更合理的策略不是二选一，而是让表示**沿着层数逐步变尺度**：

- 前层保留更细的时空分辨率，先把局部模式看清楚
- 后层逐步降低分辨率，把分散的局部线索汇聚成更大的语义单元

这其实和 CNN 里常见的层次化特征很像。前面的卷积层更多看到纹理、边缘、局部运动，后面的卷积层更多表示物体部分、整体目标和更抽象的语义。MViT 的重要之处，不是发明了“多尺度”这个想法，而是把这种本来就符合视觉规律的层次性，系统地放进了 Transformer 主干里。

这里尤其要注意一个容易混淆的点：**多尺度表示不是说同一个物体真的只存在某一个“正确尺度”，而是说模型在不同深度需要不同大小的感受范围和不同粗细的表示粒度。**

换句话说，尺度变化服务的不是“把图像缩放几次”这么简单的操作，而是让模型在前层偏向保留细节，在后层偏向整合语义。前者解决“看清楚”，后者解决“看明白”。

从这个角度再看上一节提到的那句话就更容易理解了：MViT 不是只让 token 的内容不断更新，还让 token 所代表的时空粒度也不断更新。只有这样，Transformer 才更像一个真正适合视觉任务的层次化主干，而不是把所有层都锁死在同一种分辨率上。

因此，`2.1` 的核心结论可以先记成一句话：

**视觉任务既需要前层保留细粒度局部信息，也需要后层形成大范围高语义表示，所以一个更自然的主干应当随着层数加深逐步完成从“细”到“粗”的尺度演化。**

理解了这一点，下面两个问题就顺理成章了：既然表示要从细走向粗，那么**token 数量为什么会逐步减少**，以及**通道维度为什么反而要逐步增加**？这正是下一节和下下一节要分别展开的内容。

### 2.2 Token数量为什么要逐步减少？

既然前一节已经说明，模型后层需要逐步走向更粗的尺度，那么一个直接后果就是：**token 的数量通常不能一直保持不变，而是要随着层数加深逐步减少。**

这里先不要急着把它理解成一个纯粹为了省算力的工程技巧。更根本的原因其实有两个：

- 后层不再需要为每一个细小位置都保留完全独立的表示
- 注意力对 token 数量非常敏感，尤其在视频里代价会迅速变高

先看第一个原因。

假设前层已经在很多局部 token 上提取出边缘、局部纹理、短时运动等信息，那么到了更深层，模型真正想做的事情，不再是继续把这些局部位置彼此孤立地保留下去，而是把相邻位置上的信息逐渐合并起来，形成更大的语义单元。

比如，前层也许分别看到了“手臂轮廓”“手掌局部运动”“身体朝向变化”；后层更想把它们整合成“挥手动作正在发生”。这时，如果还让每个很小的时空位置都维持一个独立 token，就会显得过细，语义组织效率也不高。

所以，token 数量减少的第一层含义是：**表示在从“位置很多、粒度很细”逐步变成“位置更少、每个 token 覆盖范围更大”。**

再看第二个原因，也就是计算问题。

在注意力里，如果序列长度记作 `L`，那么注意力矩阵的规模大致就是 `L × L`。这意味着 `L` 一旦很大，计算和显存压力都会明显上升。对图像来说，`L` 已经不算小；对视频来说，`L` 往往来自时间、宽度、高度三个维度的乘积，增长会更快。

这时如果还要求所有后续层都在原始高分辨率上做全量交互，代价通常会变得很不合理。于是，模型必须尽早把一个现实问题解决掉：

**不是所有高层语义都值得继续建立在海量细粒度 token 之上。**

也就是说，减少 token 不只是“为了快一点”，而是因为高层语义本来就更适合建立在更紧凑的表示上。计算收益和语义收益，在这里是同方向的。

这里可以把整个过程想成一种逐步汇聚：

- 前层：token 多，保留密集时空布局，方便捕捉细节
- 中层：相邻 token 开始被合并，表示更大范围的局部结构
- 后层：token 更少，但每个 token 已经承载更大区域、更长时间范围的信息

很多初学者会误以为：token 数减少，等于信息被简单删掉了。其实更准确地说，**这里发生的是“重组”和“汇聚”，而不是机械丢弃”。**

当然，下采样确实会带来一部分细节损失，但关键在于：这些细节并不是在完全没被建模时就消失，而是先在前层被提取，再在后层被压缩进更粗粒度的表示中。MViT 的设计目标，正是让这种损失尽量发生在“该压缩的时候”，而不是“还没看清就压缩”。

如果要把这一节压成一句更工程化的话，可以这样记：

**token 数量逐步减少，本质上是在让表示长度和语义层次同步收缩：前面保留足够密的时空细节，后面把这些细节汇聚成更少但更有组织的高层 token。**

这里用一张图会比纯文字更直观。

> **Nano Banana绘图提示词**  
> 画一张教学图，主题是“MViT中token数量为什么逐步减少”。横向三列布局，分别表示前层、中层、后层。左列画一个由很多小方块组成的时空token网格，中文标签“前层：token多，分辨率高，细节丰富”；中列画较少一些、每个方块更大的网格，标注“中层：相邻token开始汇聚”；右列画更少、更大的token块，标注“后层：token少，语义更集中”。每一列下方都注明 `L` 的相对变化：大、中、小；在整张图上方再画一条箭头写“层数加深”，下方再画一条说明箭头写“注意力计算压力降低”。整体风格简洁、适合深度学习课件、白底、中文标签完整、配色以蓝绿橙区分三个阶段。

理解了 token 为什么不能一直很多，下一步自然要问：既然位置数量在减少，为什么 MViT 又常常反过来把**通道维度做大**？这不是在增加表示成本吗？下一节就专门解释这个看上去有些“反直觉”的地方。

### 2.3 通道维度为什么要逐步增加？

到这里，一个很自然的疑问就是：前面不是刚刚在减少 token 数量吗？那为什么 MViT 往往又会随着层数加深，把每个 token 的通道维度逐步增加？

表面上看，这像是在一边压缩，一边扩张。但其实这两件事恰好是配套发生的。

先抓住最核心的直觉：**当 token 变少时，每个 token 通常要负责表达更大范围、更复杂的内容。**

前层一个 token 覆盖的时空区域比较小，它更多只需要描述局部边缘、局部外观、短时运动等较简单的信息；而后层一个 token 覆盖的范围更大，往往已经把多个局部区域、多帧动态线索甚至不同部位之间的关系都汇聚进来了。这样一来，如果还让它只用原来那样窄的通道维度去表达，就会显得容量不够。

换句话说，**token 数量减少，意味着“每个 token 管的地盘变大了”；通道维度增加，意味着“每个 token 的表达能力也要跟着变强”。**

这里可以用一个不那么严格但很好记的类比来理解：

- 前层像是很多基层观察员，每个人只汇报一小块区域的情况，所以每份报告可以比较短
- 后层像是更高层的汇总节点，它接收并整合了更多局部信息，所以每份表示就需要更大的“描述空间”

这也是为什么视觉主干里经常会出现一种配套规律：**分辨率下降的同时，通道数上升。** CNN 如此，MViT 也是如此。它们背后的共同逻辑是：位置维度在压缩，特征维度在增强。

这里尤其要注意一个常见误解：通道变多，并不表示模型“看到了更多位置”；位置数量的增加和减少，归属于时空分辨率或 token 数量的变化。通道变多表达的是另一件事：**对于已经保留下来的每个位置单元，模型可以用更丰富的特征维度来编码它。**

所以，`token 少了` 和 `通道多了` 根本不是互相矛盾的两个趋势，而是一种典型的容量重分配：

- 在位置维度上，不再把资源分散给过多的细粒度 token
- 在特征维度上，把更多表达能力留给更少但语义更重的 token

从张量角度也能更清楚地看这件事。假设某一层的表示形状可以粗略写成 `L × C`：

- `L` 表示 token 数量，也就是序列长度
- `C` 表示每个 token 的通道维度

MViT 在层数加深时，常见趋势是 `L` 下降、`C` 上升。它不是简单追求某一个维度单独变大或变小，而是在问：**给定更高层的语义目标，表示容量应该怎样在“有多少个 token”和“每个 token 有多强的表达能力”之间重新分配？**

于是就能看出 `2.2` 和 `2.3` 其实是一体两面：

- `2.2` 解决的是长度问题：token 不能一直太多
- `2.3` 解决的是容量问题：token 变少以后，每个 token 必须更会表达

因此，这一节的结论可以记成一句话：

**通道维度逐步增加，并不是和下采样对着干，而是在 token 逐步减少之后，对单个 token 表达能力做出的必要补偿。这样模型才能一边压缩位置数量，一边继续提升语义表示强度。**

到这里，MViT 的尺度变化机制就比较完整了：前面解释了为什么视觉建模需要从细到粗，接着说明了为什么 token 会逐步减少、通道会逐步增加。下一节就可以进一步追问：**这些尺度变化，到底是怎样被塞进注意力模块内部的？** 这正是 `Pooling Attention` 要解决的问题。